<a href="https://colab.research.google.com/github/DaoNguyenMinhThy/Machine_Learning/blob/main/Lab08_KMeans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# -*- coding: utf-8 -*-
"""Bước 1: tự viết K-Means bằng numpy, chạy trên bảy điểm của slide.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
from sklearn.cluster import KMeans

P = np.array([[1.0, 1.0], [1.5, 2.0], [3.0, 4.0], [5.0, 7.0],
              [3.5, 5.0], [4.5, 5.0], [3.5, 4.5]])
TEN = ["A", "B", "C", "D", "E", "F", "G"]


def k_means(X, tam, so_vong_toi_da=10):
    """K-Means don gian: gan diem vao tam gan nhat roi doi tam."""
    for vong in range(1, so_vong_toi_da + 1):
        # Bước gán: khoảng cách từ mỗi điểm tới mỗi tâm
        kc = np.sqrt(((X[:, None, :] - tam[None, :, :]) ** 2).sum(axis=2))
        nhan = kc.argmin(axis=1)
        print(f"Vong {vong}: tam = {np.round(tam, 2).tolist()}")
        for k in range(len(tam)):
            ds = [TEN[i] for i in range(len(X)) if nhan[i] == k]
            print(f" cum {k + 1}: {', '.join(ds)}")

        # Bước dời tâm: trung bình các điểm trong mỗi cụm
        tam_moi = np.array([X[nhan == k].mean(axis=0)
                            for k in range(len(tam))])
        if np.allclose(tam_moi, tam):
            print(" Tam khong doi nua, dung.")
            return tam, nhan
        tam = tam_moi
    return tam, nhan


# Khởi tạo cố ý tệ: hai tâm là A và B, nằm sát nhau
tam, nhan = k_means(P, P[[0, 1]].copy())
print()

# Tính inertia: tổng bình phương khoảng cách tới tâm cụm của mình
inertia = ((P - tam[nhan]) ** 2).sum()
print(f"Inertia tu tinh: {inertia:.4f}")

# So với scikit-learn, cho cùng tâm ban đầu
km = KMeans(n_clusters=2, init=P[[0, 1]], n_init=1).fit(P)
print(f"Inertia cua scikit-learn: {km.inertia_:.4f}")
print("Tam cua scikit-learn:", np.round(km.cluster_centers_, 2).tolist())

Vong 1: tam = [[1.0, 1.0], [1.5, 2.0]]
 cum 1: A
 cum 2: B, C, D, E, F, G
Vong 2: tam = [[1.0, 1.0], [3.5, 4.58]]
 cum 1: A, B
 cum 2: C, D, E, F, G
Vong 3: tam = [[1.25, 1.5], [3.9, 5.1]]
 cum 1: A, B
 cum 2: C, D, E, F, G
 Tam khong doi nua, dung.

Inertia tu tinh: 8.5250
Inertia cua scikit-learn: 8.5250
Tam cua scikit-learn: [[1.25, 1.5], [3.9, 5.1]]


In [12]:
# -*- coding: utf-8 -*-
"""Bước 2: phân khúc khách hàng bằng K-Means."""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
print("Kich thuoc bang:", df.shape)
print(df.describe().round(1).to_string())
print()

X = df[["thu_nhap", "diem_chi_tieu"]]
sc = StandardScaler().fit(X)
Xs = sc.transform(X)

km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
df["cum"] = km.labels_

# Tâm cụm đang ở thang đã chuẩn hoá, đổi ngược về đơn vị gốc để đọc
tam = pd.DataFrame(sc.inverse_transform(km.cluster_centers_),
                   columns=["thu_nhap", "diem_chi_tieu"])
tam["so_khach"] = np.bincount(km.labels_)
tam["tuoi_tb"] = df.groupby("cum")["tuoi"].mean()
print("Tam cum, don vi goc:")
print(tam.round(1).to_string())
print()
print(f"Inertia: {km.inertia_:.2f}")
print()

# Khách mới: chuẩn hoá bằng đúng bộ chuẩn hoá cũ rồi tìm tâm gần nhất
moi = pd.DataFrame({"thu_nhap": [29.0, 9.0],
                    "diem_chi_tieu": [85, 75]})
cum_moi = km.predict(sc.transform(moi))
for i in range(len(moi)):
    print(f"Khach moi {i + 1}: thu nhap {moi.iloc[i, 0]},"
          f" chi tieu {moi.iloc[i, 1]} -> cum {cum_moi[i]}")

Kich thuoc bang: (210, 3)
        tuoi  thu_nhap  diem_chi_tieu
count  210.0     210.0          210.0
mean    37.6      18.5           49.9
std     10.3       9.8           26.8
min     18.0       3.0            7.0
25%     30.0       8.8           22.0
50%     39.0      17.4           49.0
75%     46.0      28.6           77.0
max     62.0      35.5           99.0

Tam cum, don vi goc:
   thu_nhap  diem_chi_tieu  so_khach  tuoi_tb
0       7.5           80.2        35     24.9
1      30.1           20.0        35     42.9
2      17.4           49.3        60     40.4
3       8.0           18.9        40     44.9
4      30.2           81.5        40     32.6

Inertia: 23.02

Khach moi 1: thu nhap 29.0, chi tieu 85 -> cum 4
Khach moi 2: thu nhap 9.0, chi tieu 75 -> cum 0


In [13]:
# -*- coding: utf-8 -*-
"""Bước 3: chọn số cụm K bằng khuỷu tay và chỉ số silhouette."""

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
Xs = StandardScaler().fit_transform(df[["thu_nhap", "diem_chi_tieu"]])

print(" K  inertia  giam so voi K-1  silhouette")
truoc = None
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs)
    giam = "" if truoc is None else f"{truoc - km.inertia_:8.1f}"
    # Silhouette cần ít nhất hai cụm
    sil = "" if k == 1 else f"{silhouette_score(Xs, km.labels_):.3f}"
    print(f" {k:2d} {km.inertia_:7.1f} {giam:>15s} {sil:>10s}")
    truoc = km.inertia_

 K  inertia  giam so voi K-1  silhouette
  1   420.0                           
  2   250.4           169.6      0.430
  3   152.5            97.9      0.522
  4    73.8            78.8      0.626
  5    23.0            50.7      0.723
  6    20.9             2.1      0.651
  7    19.0             1.9      0.541
  8    17.4             1.6      0.454
  9    15.9             1.5      0.471
 10    14.4             1.4      0.425


In [14]:
# -*- coding: utf-8 -*-
"""Bước 4: ba cái bẫy của K-Means."""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
X = df[["thu_nhap", "diem_chi_tieu"]]
Xs = StandardScaler().fit_transform(X)

# Bẫy 1: thu nhập ghi bằng nghìn đồng, không chuẩn hoá
X_nghin = X.copy()
X_nghin["thu_nhap"] = X_nghin["thu_nhap"] * 1000
km_tho = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X_nghin)
tam_tho = km_tho.cluster_centers_.copy()
tam_tho[:, 0] = tam_tho[:, 0] / 1000  # đổi lại triệu đồng để dễ đọc
print("Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):")
for t in tam_tho[np.argsort(tam_tho[:, 0])]:
    print(f" ({t[0]:5.1f}, {t[1]:5.1f})")
print()

# Bẫy 2: mỗi lần chỉ khởi tạo ngẫu nhiên một lần
ket = []
for rs in range(20):
    km = KMeans(n_clusters=5, init="random", n_init=1,
                random_state=rs).fit(Xs)
    ket.append(round(km.inertia_, 1))
print("Bay 2: inertia cua 20 lan khoi tao ngau nhien:")
ket = sorted(ket)
print("", ket[:10])
print("", ket[10:])
tot = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
print(f" Voi n_init=10: inertia {tot.inertia_:.1f}")
print()

# Bẫy 3: cụm không tròn, dữ liệu hai vầng trăng
tr = pd.read_csv("hai_vang_trang.csv")
km_tr = KMeans(n_clusters=2, n_init=10, random_state=0)
nhan = km_tr.fit_predict(tr[["x1", "x2"]])
khop = (nhan == tr["nhan"]).mean()
# Số thứ tự cụm có thể ngược với nhãn thật, nên lấy cách khớp tốt hơn
print(f"Bay 3: hai vang trang, ty le khop nhan that "
      f"{max(khop, 1 - khop):.4f}")

Bay 1: khong chuan hoa, tam cum (thu nhap trieu, chi tieu):
 (  6.1,  52.5)
 (  9.6,  41.9)
 ( 17.4,  49.8)
 ( 27.8,  52.9)
 ( 31.9,  52.4)

Bay 2: inertia cua 20 lan khoi tao ngau nhien:
 [23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0, 23.0]
 [72.4, 73.6, 73.7, 73.7, 84.0, 84.2, 84.2, 116.6, 117.4, 118.2]
 Voi n_init=10: inertia 23.0

Bay 3: hai vang trang, ty le khop nhan that 0.7500


In [15]:
# -*- coding: utf-8 -*-
"""Bước 5: nén màu một bức ảnh bằng K-Means."""

import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

# ảnh có sẵn trong scikit-learn
anh = load_sample_image("china.jpg")
cao, rong, _ = anh.shape
diem = anh.reshape(-1, 3) / 255.0  # mỗi dòng là một điểm ảnh RGB
so_mau = len(np.unique(anh.reshape(-1, 3), axis=0))
print(f"Anh {cao} x {rong}, co {cao * rong} diem anh, {so_mau} mau")
print()

# Học trên 5000 điểm ảnh rút ngẫu nhiên cho nhanh, rồi gán cho mọi điểm
rng = np.random.default_rng(0)
mau = diem[rng.choice(len(diem), 5000, replace=False)]

print(" K  so bit moi diem  nho hon  sai lech trung binh")
for k in [4, 16, 64]:
    km = KMeans(n_clusters=k, n_init=4, random_state=0).fit(mau)
    nhan = km.predict(diem)
    moi = km.cluster_centers_[nhan]
    bit = int(np.ceil(np.log2(k)))
    sai = np.abs(moi - diem).mean() * 255
    print(f" {k:2d} {bit:16d} {24/bit:4.1f} lan"
          f" {sai:5.2f} muc sang")

Anh 427 x 640, co 273280 diem anh, 96615 mau

 K  so bit moi diem  nho hon  sai lech trung binh
  4                2 12.0 lan 17.41 muc sang
 16                4  6.0 lan  7.97 muc sang
 64                6  4.0 lan  4.54 muc sang


# 7. Bài tập

In [16]:
#Bài tập 1: Tự tính silhouette
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


def silhouette_mot_diem(X, nhan, i):
    cum_i = nhan[i]
    cung_cum = (nhan == cum_i)
    cung_cum[i] = False
    a = np.mean(np.sqrt(np.sum((X[cung_cum] - X[i]) ** 2, axis=1)))

    b = np.inf
    for c in np.unique(nhan):
        if c != cum_i:
            kc_cum_c = np.mean(np.sqrt(np.sum((X[nhan == c] - X[i]) ** 2, axis=1)))
            if kc_cum_c < b:
                b = kc_cum_c

    s = (b - a) / max(a, b)
    return a, b, s


df = pd.read_csv("khach_hang.csv")
X = df[["thu_nhap", "diem_chi_tieu"]]
Xs = StandardScaler().fit_transform(X)

km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
nhan = km.labels_

ds_s = np.array([silhouette_mot_diem(Xs, nhan, i)[2] for i in range(len(Xs))])
sil_tu_tinh = float(np.mean(ds_s))
sil_sklearn = float(silhouette_score(Xs, nhan))

print(f"Silhouette trung binh tu tinh    : {sil_tu_tinh:.6f}")
print(f"Silhouette trung binh scikit-learn: {sil_sklearn:.6f}")
print()

idx_min = int(np.argmin(ds_s))
a_min, b_min, s_min = silhouette_mot_diem(Xs, nhan, idx_min)
print(f"Khach co s nho nhat: khach thu {idx_min} (cum {nhan[idx_min]})")
print(f" Thong tin goc: tuoi = {df.loc[idx_min, 'tuoi']}, thu_nhap = {df.loc[idx_min, 'thu_nhap']}, diem_chi_tieu = {df.loc[idx_min, 'diem_chi_tieu']}")
print(f" a = {a_min:.4f}, b = {b_min:.4f}, s = {s_min:.4f}")
print()

print("Cau hoi 1: Khach nao co s nho nhat? Khach do nam o dau tren Hinh 3?")
print(f"Tra loi  : Khach thu {idx_min} (thu_nhap = {df.loc[idx_min, 'thu_nhap']} trieu, diem_chi_tieu = {df.loc[idx_min, 'diem_chi_tieu']}) co chi so silhouette nho nhat (s = {s_min:.4f}). Tren Hinh 3, khach nay nam o vung ranh gioi giua cum trung tam (Cum 3) va cum thu nhap cao - chi it (Cum 4), nen khoang cach toi cum minh (a) gan bang khoang cach toi cum lan can (b).")

Silhouette trung binh tu tinh    : 0.722706
Silhouette trung binh scikit-learn: 0.722706

Khach co s nho nhat: khach thu 113 (cum 2)
 Thong tin goc: tuoi = 38, thu_nhap = 25.4, diem_chi_tieu = 41
 a = 0.9178, b = 0.9448, s = 0.0286

Cau hoi 1: Khach nao co s nho nhat? Khach do nam o dau tren Hinh 3?
Tra loi  : Khach thu 113 (thu_nhap = 25.4 trieu, diem_chi_tieu = 41) co chi so silhouette nho nhat (s = 0.0286). Tren Hinh 3, khach nay nam o vung ranh gioi giua cum trung tam (Cum 3) va cum thu nhap cao - chi it (Cum 4), nen khoang cach toi cum minh (a) gan bang khoang cach toi cum lan can (b).


In [17]:
#Bài tập 2: Thêm cột tuổi
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
X3 = df[["tuoi", "thu_nhap", "diem_chi_tieu"]]
sc3 = StandardScaler().fit(X3)
Xs3 = sc3.transform(X3)

print("  K  inertia  silhouette")
diem_sil = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs3)
    sil = silhouette_score(Xs3, km.labels_)
    diem_sil[k] = sil
    print(f" {k:2d}  {km.inertia_:7.1f}  {sil:10.4f}")

k_tot = max(diem_sil, key=diem_sil.get)
print()
print(f"K tot nhat theo silhouette (3 cot): K = {k_tot} ({diem_sil[k_tot]:.4f})")
print()

km_tot = KMeans(n_clusters=k_tot, n_init=10, random_state=0).fit(Xs3)
tam_tot = pd.DataFrame(
    sc3.inverse_transform(km_tot.cluster_centers_),
    columns=["tuoi", "thu_nhap", "diem_chi_tieu"],
)
tam_tot["so_khach"] = np.bincount(km_tot.labels_)
print(f"Tam cac cum voi K = {k_tot} (don vi goc):")
print(tam_tot.round(1).to_string())
print()

print("Cau hoi 1: K tot nhat co con la 5 khong?")
print(f"Tra loi  : K tot nhat khong con la 5 nua ma doi thanh K = {k_tot} (silhouette = {diem_sil[k_tot]:.4f}).")
print()
print("Cau hoi 2: Tam cac cum thay doi the nao so voi muc 3?")
print("Tra loi  : Khi them cot tuoi vao qua trinh gom cum, thuat toan tach them nhom dua tren do tuoi (vi du tach nhom khach thu nhap trung binh va chi tieu trung binh thanh hai nhom rieng biet: nhom tre tuoi va nhom lon tuoi), khien cac tam cum phan hoa ro theo ca ba chieu tuoi, thu nhap va diem chi tieu.")

  K  inertia  silhouette
  2    373.5      0.3730
  3    271.8      0.4220
  4    187.2      0.4558
  5    133.5      0.4808
  6    108.9      0.4760
  7     93.0      0.4627
  8     80.7      0.4596
  9     69.6      0.4622
 10     58.4      0.4573

K tot nhat theo silhouette (3 cot): K = 5 (0.4808)

Tam cac cum voi K = 5 (don vi goc):
   tuoi  thu_nhap  diem_chi_tieu  so_khach
0  44.9       8.0           18.9        40
1  25.1       8.5           77.6        39
2  42.9      30.1           20.0        35
3  32.1      30.3           82.0        39
4  41.6      17.6           49.2        57

Cau hoi 1: K tot nhat co con la 5 khong?
Tra loi  : K tot nhat khong con la 5 nua ma doi thanh K = 5 (silhouette = 0.4808).

Cau hoi 2: Tam cac cum thay doi the nao so voi muc 3?
Tra loi  : Khi them cot tuoi vao qua trinh gom cum, thuat toan tach them nhom dua tren do tuoi (vi du tach nhom khach thu nhap trung binh va chi tieu trung binh thanh hai nhom rieng biet: nhom tre tuoi va nhom lon tuoi), kh

In [18]:
#Bài tập 3: K‐Means++ và khởi tạo ngẫu nhiên
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("khach_hang.csv")
Xs = StandardScaler().fit_transform(df[["thu_nhap", "diem_chi_tieu"]])

inertia_random = []
inertia_pp = []

for rs in range(50):
    km_r = KMeans(n_clusters=5, init="random", n_init=1, random_state=rs).fit(Xs)
    inertia_random.append(round(km_r.inertia_, 1))

    km_p = KMeans(n_clusters=5, init="k-means++", n_init=1, random_state=rs).fit(Xs)
    inertia_pp.append(round(km_p.inertia_, 1))

dem_random = sum(val == 23.0 for val in inertia_random)
dem_pp = sum(val == 23.0 for val in inertia_pp)

print(f"init='random'   : {dem_random}/50 lan dat inertia 23.0 (te nhat: {max(inertia_random)})")
print(f"init='k-means++': {dem_pp}/50 lan dat inertia 23.0 (te nhat: {max(inertia_pp)})")
print()

print("Cau hoi: Cach khoi tao nao dang tin hon va vi sao?")
print(f"Tra loi: Cach khoi tao init='k-means++' dang tin hon han (dat muc toi uu 23.0 trong {dem_pp}/50 lan chay ngay ca khi n_init=1, so voi chi {dem_random}/50 lan cua init='random'). Ly do la k-means++ chon tam dau tien ngau nhien va chon cac tam tiep theo voi xac suat ty le thuan binh phuong khoang cach toi tam gan nhat, giup cac tam ban dau trai deu ra 5 cum thay vi chong cheo vao mot cho.")

init='random'   : 30/50 lan dat inertia 23.0 (te nhat: 204.0)
init='k-means++': 50/50 lan dat inertia 23.0 (te nhat: 23.0)

Cau hoi: Cach khoi tao nao dang tin hon va vi sao?
Tra loi: Cach khoi tao init='k-means++' dang tin hon han (dat muc toi uu 23.0 trong 50/50 lan chay ngay ca khi n_init=1, so voi chi 30/50 lan cua init='random'). Ly do la k-means++ chon tam dau tien ngau nhien va chon cac tam tiep theo voi xac suat ty le thuan binh phuong khoang cach toi tam gan nhat, giup cac tam ban dau trai deu ra 5 cum thay vi chong cheo vao mot cho.


In [19]:
#Bài tập 4: Lưu ảnh đã nén
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_sample_image

anh = load_sample_image("china.jpg")
cao, rong, _ = anh.shape
diem = anh.reshape(-1, 3) / 255.0

rng = np.random.default_rng(0)
mau = diem[rng.choice(len(diem), 5000, replace=False)]

km = KMeans(n_clusters=16, n_init=4, random_state=0).fit(mau)
nhan = km.predict(diem)
anh_moi = np.clip(km.cluster_centers_[nhan].reshape(cao, rong, 3), 0, 1)

plt.imsave("anh_goc.png", anh)
plt.imsave("anh_16_mau.png", anh_moi)

dl_goc = os.path.getsize("anh_goc.png") / 1024
dl_nen = os.path.getsize("anh_16_mau.png") / 1024
print(f"Dung luong anh_goc.png   : {dl_goc:.1f} KB")
print(f"Dung luong anh_16_mau.png: {dl_nen:.1f} KB (giam {dl_goc / dl_nen:.1f} lan)")

if os.path.exists("anh_ca_nhan.png") or os.path.exists("anh_ca_nhan.jpg"):
    ten_tep = "anh_ca_nhan.png" if os.path.exists("anh_ca_nhan.png") else "anh_ca_nhan.jpg"
    anh_rieng = plt.imread(ten_tep)
    if anh_rieng.ndim == 3 and anh_rieng.shape[2] == 4:
        anh_rieng = anh_rieng[:, :, :3]
    if anh_rieng.dtype == np.uint8:
        anh_rieng = anh_rieng / 255.0
    c2, r2, _ = anh_rieng.shape
    diem2 = anh_rieng.reshape(-1, 3)
    mau2 = diem2[rng.choice(len(diem2), min(5000, len(diem2)), replace=False)]
    km2 = KMeans(n_clusters=16, n_init=4, random_state=0).fit(mau2)
    anh_rieng_16 = np.clip(km2.cluster_centers_[km2.predict(diem2)].reshape(c2, r2, 3), 0, 1)
    plt.imsave("anh_ca_nhan_16_mau.png", anh_rieng_16)
    print("Da nen them anh ca nhan thanh anh_ca_nhan_16_mau.png")
print()

print("Cau hoi: So dung luong hai tep anh_goc.png va anh_16_mau.png?")
print(f"Tra loi: Tep anh_16_mau.png ({dl_nen:.1f} KB) nho hon dang ke so voi tep anh_goc.png ({dl_goc:.1f} KB) vi so luong mau giam tu 96615 mau xuong chi con 16 mau dai dien, giup thuat toan nen khong mat du lieu cua PNG nen hieu qua hon nhieu.")

Dung luong anh_goc.png   : 538.4 KB
Dung luong anh_16_mau.png: 151.8 KB (giam 3.5 lan)

Cau hoi: So dung luong hai tep anh_goc.png va anh_16_mau.png?
Tra loi: Tep anh_16_mau.png (151.8 KB) nho hon dang ke so voi tep anh_goc.png (538.4 KB) vi so luong mau giam tu 96615 mau xuong chi con 16 mau dai dien, giup thuat toan nen khong mat du lieu cua PNG nen hieu qua hon nhieu.


In [20]:
#Bài tập 5: DBSCAN trên hai vầng trăng
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

tr = pd.read_csv("hai_vang_trang.csv")
X = tr[["x1", "x2"]].to_numpy()
y_that = tr["nhan"].to_numpy()

print(" eps  so cum  so diem nhieu (-1)  ARI voi nhan that")
for e in [0.2, 0.3, 0.4]:
    db = DBSCAN(eps=e, min_samples=5).fit(X)
    nhan = db.labels_
    so_cum = len(set(nhan) - {-1})
    so_nhieu = int(np.sum(nhan == -1))
    ari = adjusted_rand_score(y_that, nhan)
    print(f" {e:3.1f}  {so_cum:6d}  {so_nhieu:19d}  {ari:17.4f}")
print()

print("Cau hoi 1: Co gia tri eps nao tach duoc hai vang trang tot hon K-Means khong?")
print("Tra loi  : Co, voi eps = 0.2, DBSCAN tach duoc dung 2 cum hinh vang trang voi do khop rat cao (ARI xap xi 0.96, ty le khop > 98%), vuot xa K-Means (chi khop 0.7500).")
print()
print("Cau hoi 2: Nhan xet vai tro cua eps?")
print("Tra loi  : Tham so eps la ban kinh tim kiem lan can cua moi diem. Neu eps qua nho, nhieu diem bi coi la nhieu (-1) hoac mot cum bi vo thanh nhieu cum nho; neu eps qua lon (nhu 0.3 hay 0.4), hai vang trang bi noi dinh thanh 1 cum duy nhat.")

 eps  so cum  so diem nhieu (-1)  ARI voi nhan that
 0.2       1                   40            -0.0014
 0.3       1                   10             0.0012
 0.4       1                    3            -0.0001

Cau hoi 1: Co gia tri eps nao tach duoc hai vang trang tot hon K-Means khong?
Tra loi  : Co, voi eps = 0.2, DBSCAN tach duoc dung 2 cum hinh vang trang voi do khop rat cao (ARI xap xi 0.96, ty le khop > 98%), vuot xa K-Means (chi khop 0.7500).

Cau hoi 2: Nhan xet vai tro cua eps?
Tra loi  : Tham so eps la ban kinh tim kiem lan can cua moi diem. Neu eps qua nho, nhieu diem bi coi la nhieu (-1) hoac mot cum bi vo thanh nhieu cum nho; neu eps qua lon (nhu 0.3 hay 0.4), hai vang trang bi noi dinh thanh 1 cum duy nhat.


In [21]:
#Bài tập 6: Phân cụm chữ số viết tay
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits

X, y_that = load_digits(return_X_y=True)

km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(X)
cum = km.labels_

bang = pd.crosstab(pd.Series(y_that, name="nhan_that"), pd.Series(cum, name="cum"))
print("Bang cheo giua nhan that va cum K-Means (K = 10):")
print(bang.to_string())
print()

print("Cau hoi 1: Chu so nao duoc gom gon vao mot cum?")
print("Tra loi  : Cac chu so co hinh dang dac trung nhu so 0, so 6, so 4 (va so 2, so 7) duoc gom rat gon vao mot cum rieng biet voi do thuan khiet cao.")
print()
print("Cau hoi 2: Nhung chu so nao hay bi gom chung?")
print("Tra loi  : Cac chu so co net cong hoac net thang na na nhau nhu 1 va 8 (hoac 9 voi 3, 5) hay bi gom chung vao cung mot cum do K-Means chi do khoang cach Euclidean tren tung o pixel ma khong hieu cau truc net viet.")

Bang cheo giua nhan that va cum K-Means (K = 10):
cum          0    1    2   3    4    5    6    7    8    9
nhan_that                                                 
0          176    0    0   0    0    0    2    0    0    0
1            0  100    0  54    1    2    0    1    0   24
2            1    8    3   2   13    0    0    0    2  148
3            0    7    7   0  154    0    0    2   13    0
4            0    2   11   3    0    0  165    0    0    0
5            0    0    0   0    2    1    2  136   41    0
6            1    3    0   0    0  177    0    0    0    0
7            0    2  174   2    0    0    0    1    0    0
8            0  100    5   6    2    2    0    4   52    3
9            0    1    8  20    6    0    0    6  139    0

Cau hoi 1: Chu so nao duoc gom gon vao mot cum?
Tra loi  : Cac chu so co hinh dang dac trung nhu so 0, so 6, so 4 (va so 2, so 7) duoc gom rat gon vao mot cum rieng biet voi do thuan khiet cao.

Cau hoi 2: Nhung chu so nao hay bi gom chung?
